<a href="https://colab.research.google.com/github/sathishweb997/ai-mentor-portfolio-sathishlutukurthi/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Gemini API key: ··········


In [3]:
!pip install -U ddgs
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.9 MB/s eta 0:00:00
Apply online for TCS Recruitment 2026. Latest TCS Smart Hiring, BPS jobs, IT jobs, and engineering vacancies for freshers and experienced candidates. Register for TCS NQT 2026. Check eligibility, salary, exam pattern, and selection process. Open for 2021 to 2027 batches. Apply For TCS NQT 2026 Hiring before May 14th, 2026. TCS Hiring 2026 is live with massive job openings for freshers and experien


In [4]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_2068/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [5]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'bcc0572d-101a-4539-9e06-3318f36059fe', 'type': 'tool_call'}]

[2] ToolMessage
    Content: 24 feb 2026 · TCS has opened Engineering Fresher Hiring through the National Qualifier Test (NQT) for Digital & Prime roles. Eligible batches: 2024 | 2025 | 2026. Degrees: ... 18 feb 2026 · Start preparing and take that first big step towards your dream job. Bye bye. OCR. Freshers Hiring TCS TATA CO

[3] AIMessage
    Content: [{'type': 'text', 'text': "I couldn't find a specific hiring quota for TCS in 2026 in my search results. The information available indicates that TCS has opened various hiring drives for the 2026 batch, including through the National Qualifier Test (NQT) and Smart Hiring programs, with application d


In [6]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I cannot directly access the content of a URL. The `web_search` tool can search the web for information based on a query. Also, the URL you provided appears to be invalid.\n\nIf you are looking for specific information, please provide a search query instead of a URL.', 'ex


In [7]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [8]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))
# Expected: 'aws, spring boot'

aws, spring boot


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Expected: low score (~3-4/10), rationale about lack of specificity / cultural fit

**Score: 2/10**

**Rationale:** Highly transactional and superficial, showing no genuine interest in the company's work or culture beyond basic benefits.


In [10]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_2068/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [10]:
import json, pathlib

# Create the data folder and profiles file
pathlib.Path('data').mkdir(exist_ok=True)

profiles = [
    {
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 8.2,
        "target_company": "TCS Digital",
        "skills": ["Python", "Java", "SQL", "Data Structures"]
    },
    {
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 7.8,
        "target_company": "Cognizant",
        "skills": ["Python", "C++", "Communication", "Problem Solving"]
    },
    {
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.9,
        "target_company": "Amazon",
        "skills": ["Python", "Java", "AWS", "SQL", "Spring Boot", "System Design"]
    }
]

pathlib.Path('data/student_profiles.json').write_text(json.dumps(profiles, indent=2))
print('Profiles saved!')

Profiles saved!
